# Tamper-Evident Audit Trail using Intervention Handler and asqav

This cookbook shows how to use an intervention handler to log all agent messages
and tool calls to a tamper-evident audit trail using [asqav](https://github.com/jagmarques/asqav-sdk).

asqav provides cryptographically signed audit trails for AI agents using
ML-DSA (FIPS 204). Each action is signed server-side and chained to form
a verifiable record of agent behavior.

```{note}
This recipe uses {py:class}`~autogen_core.SingleThreadedAgentRuntime`,
which supports intervention handlers.
```

## Installation

Install the required packages:

```bash
pip install asqav autogen-core
```

You will need an asqav API key from [asqav.com](https://asqav.com).

In [ ]:
import json
from dataclasses import dataclass
from typing import Any

import asqav
from autogen_core import (
    AgentId,
    DefaultInterventionHandler,
    DefaultTopicId,
    FunctionCall,
    MessageContext,
    RoutedAgent,
    SingleThreadedAgentRuntime,
    default_subscription,
    message_handler,
)

## Define message types

We define a simple message type for this example.

In [ ]:
@dataclass
class Message:
    content: str

## Create the AsqavInterventionHandler

The handler subclasses {py:class}`~autogen_core.DefaultInterventionHandler` and
signs each message and response to asqav's audit trail. It intercepts three
lifecycle hooks:

- `on_send` - direct messages between agents
- `on_publish` - broadcast messages to topics
- `on_response` - responses returned from agent message handlers

If a message contains a {py:class}`~autogen_core.FunctionCall`, the handler
logs the tool name and arguments separately for fine-grained auditing.

In [ ]:
class AsqavInterventionHandler(DefaultInterventionHandler):
    """Intervention handler that logs all agent messages and tool calls
    to a tamper-evident audit trail using asqav."""

    def __init__(self, asqav_agent: asqav.Agent) -> None:
        self._agent = asqav_agent

    async def on_send(
        self, message: Any, *, message_context: MessageContext, recipient: AgentId
    ) -> Any:
        action_data = {
            "recipient": str(recipient),
            "message_type": type(message).__name__,
        }
        # Log tool calls with additional detail.
        if isinstance(message, FunctionCall):
            action_data["tool_name"] = message.name
            action_data["tool_arguments"] = message.arguments
            self._agent.sign("agent:tool_call", action_data)
        else:
            self._agent.sign("agent:send", action_data)
        return message

    async def on_publish(
        self, message: Any, *, message_context: MessageContext
    ) -> Any:
        action_data = {
            "message_type": type(message).__name__,
        }
        self._agent.sign("agent:publish", action_data)
        return message

    async def on_response(
        self, message: Any, *, sender: AgentId, recipient: AgentId | None
    ) -> Any:
        action_data = {
            "sender": str(sender),
            "recipient": str(recipient) if recipient else None,
            "message_type": type(message).__name__,
        }
        self._agent.sign("agent:response", action_data)
        return message

## Create a sample agent

A minimal agent that echoes messages back, so we can demonstrate the audit trail.

In [ ]:
@default_subscription
class EchoAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("EchoAgent")

    @message_handler
    async def on_message(self, message: Message, ctx: MessageContext) -> None:
        print(f"EchoAgent received: {message.content}")

## Wire up the runtime

Initialize asqav, create the intervention handler, and register it with the runtime.

In [ ]:
# Initialize asqav with your API key.
asqav.init(api_key="sk_...")
audit_agent = asqav.Agent.create("autogen-auditor")

# Create the runtime with the asqav intervention handler.
handler = AsqavInterventionHandler(asqav_agent=audit_agent)
runtime = SingleThreadedAgentRuntime(intervention_handlers=[handler])

# Register the agent.
await EchoAgent.register(runtime, "echo_agent", EchoAgent)

runtime.start()

# Publish messages - each one is signed to the audit trail.
await runtime.publish_message(Message("Hello from user"), DefaultTopicId())
await runtime.publish_message(Message("Another message"), DefaultTopicId())

await runtime.stop()

Every message that passes through the runtime is now signed with ML-DSA and
recorded in asqav's tamper-evident log. You can view the full audit trail
and verify individual signatures at [asqav.com/dashboard](https://asqav.com/dashboard).

## What gets logged

| Hook | Action type | Details |
|---|---|---|
| `on_send` | `agent:send` | Recipient, message type |
| `on_send` (tool call) | `agent:tool_call` | Tool name, arguments, recipient |
| `on_publish` | `agent:publish` | Message type |
| `on_response` | `agent:response` | Sender, recipient, message type |

Each signed action returns a `signature_id`, `chain_hash`, and `verify_url`
that can be used for compliance reporting and independent verification.

## Further reading

- [asqav documentation](https://asqav.com/docs)
- [asqav SDK on PyPI](https://pypi.org/project/asqav/)
- [InterventionHandler API reference](https://microsoft.github.io/autogen/dev/reference/python/autogen_core.html#autogen_core.DefaultInterventionHandler)